In [5]:
# importazione delle librerie necessarie

import requests
import urllib3
import numpy as np
from tensorflow import keras

# elimanre gli avertimenti di sicurezza
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("Librerie importate correttamente!")

Librerie importate correttamente!


In [6]:
# Download della Divina Commedia
def scarica_testo_dante():
    """
    Scarica la Divina Commedia da internet.
    Prova prima il link principale, poi uno di riserva.
    """
    url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
    print(f"Scaricamento testo da: {url}...")
    
    try:
        response = requests.get(url, verify=False)
        response.raise_for_status()
        response.encoding = 'utf-8'
        text = response.text
        print(f"Download completato! Lunghezza testo: {len(text)} caratteri")
        print(f"\nEsempio (primi 100 caratteri):\n{text[:100]}")
        return text
    except Exception as e:
        print(f"Errore: {e}")
        print("Tentativo con link di riserva (GitHub)...")
        try:
            url_backup = "https://raw.githubusercontent.com/wpm/t-sne-text-vis/master/data/divina_commedia.txt"
            r2 = requests.get(url_backup, verify=False)
            r2.encoding = 'utf-8'
            return r2.text
        except:
            return None

# Eseguiamo il download
testo = scarica_testo_dante()

Scaricamento testo da: https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt...
Download completato! Lunghezza testo: 551846 caratteri

Esempio (primi 100 caratteri):
LA DIVINA COMMEDIA
di Dante Alighieri
INFERNO



Inferno: Canto I

  Nel mezzo del cammin di


In [7]:
# Elaborazione del testo (elabora_dizionari)
def elabora_dizionari(testo):
    """
    Trova tutti i caratteri unici nel testo e crea
    dei dizionari per convertire caratteri <-> numeri.
    La rete neurale lavora con numeri, non con lettere!
    """
    caratteri_unici = sorted(set(testo))
    num_caratteri_unici = len(caratteri_unici)
    
    print(f"Caratteri unici trovati: {num_caratteri_unici}")
    print(f"Esempio caratteri: {caratteri_unici[:20]}")
    
    # Dizionario: carattere -> numero (es. 'a' -> 5)
    char_to_idx = {c: i for i, c in enumerate(caratteri_unici)}
    
    # Dizionario: numero -> carattere (es. 5 -> 'a')
    idx_to_char = {i: c for i, c in enumerate(caratteri_unici)}
    
    return caratteri_unici, num_caratteri_unici, char_to_idx, idx_to_char

# Eseguiamo
caratteri, num_car, char_to_idx, idx_to_char = elabora_dizionari(testo)

Caratteri unici trovati: 69
Esempio caratteri: ['\n', '\r', ' ', '!', '"', "'", '(', ')', ',', '-', '.', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F']


In [8]:
# Creazione del dataset di addestramento
def crea_dataset_addestramento(testo, char_to_idx, num_car):
    """
    Prepara gli 'esercizi' per la rete neurale.
    Prende sequenze di 80 caratteri e chiede: quale viene dopo?
    Es: "Nel mezzo del cammin di nostr" -> 'a'
    """
    seq_length = 80  # lunghezza di ogni sequenza
    
    X = []  # input: sequenze di 80 caratteri
    y = []  # output: il carattere successivo
    
    for i in range(0, len(testo) - seq_length):
        sequenza = testo[i:i + seq_length]
        carattere_successivo = testo[i + seq_length]
        
        X.append([char_to_idx[c] for c in sequenza])
        y.append(char_to_idx[carattere_successivo])
    
    print(f"Numero di sequenze create: {len(X)}")
    
    # Convertiamo in array numpy e normalizziamo
    X = np.reshape(X, (len(X), seq_length, 1))
    X = X / num_car  # valori tra 0 e 1
    y = keras.utils.to_categorical(y, num_classes=num_car)
    
    print(f"Shape X: {X.shape}")
    print(f"Shape y: {y.shape}")
    
    return X, y

# Eseguiamo
X, y = crea_dataset_addestramento(testo, char_to_idx, num_car)

Numero di sequenze create: 551766
Shape X: (551766, 80, 1)
Shape y: (551766, 69)
